In [2]:
import boto3
import json

bedrock = boto3.client(service_name='bedrock-runtime')
bedrock_agent = boto3.client(service_name='bedrock-agent')

MODEL_ID = "amazon.nova-micro-v1:0"

Traveling with Bedrock

In [4]:
import boto3
import json

REGION = "us-east-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

# Use an inference profile ID if raw model ID fails in your account/region
MODEL_ID = "us.amazon.nova-micro-v1:0"

temperature = 0.7
inference_config = {
    "temperature": temperature,
    "maxTokens": 300
}

system_prompts = [
    {
        "text": (
            "You are a virtual travel assistant that suggests destinations "
            "based on user preferences. Only return destination names and a brief description."
        )
    }
]

messages = [
    {
        "role": "user",
        "content": [{"text": "Create a list of 3 travel destinations."}]
    }
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

def print_response(response):
    content = response.get("output", {}).get("message", {}).get("content", [])
    text_parts = [part["text"] for part in content if "text" in part]

    print("✈️ Your suggested travel destinations:")
    print("\n".join(text_parts))

print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan**  
   Experience traditional Japanese culture with stunning temples, serene gardens, and the famous cherry blossoms.

2. **Santorini, Greece**  
   Enjoy picturesque views, white-washed buildings, and beautiful sunsets over the Aegean Sea on this iconic Greek island.

3. **Cape Town, South Africa**  
   Discover stunning natural beauty with Table Mountain, vibrant markets, and rich cultural heritage.


Continuing the conversation

In [5]:
message_2 = {
        "role": "user",
        "content": [{"text": "Only suggest travel locations that are no more than one short flight away."}]
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Orlando, Florida**  
   Description: Known for its world-famous theme parks, including Walt Disney World, Universal Studios, and SeaWorld.

2. **Miami, Florida**  
   Description: A vibrant city with stunning beaches, colorful street art, and a rich Latin American culture.

3. **Nashville, Tennessee**  
   Description: The heart of country music, featuring live music venues along Broadway, and the iconic Grand Ole Opry.


Your turn to refine the results

In [6]:
message_3 = {
        "role": "user",
        "content": [{"text": "INSERT YOUR PROMPT HERE"}]
}

messages.append(message_3)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **San Diego, California**
   - Known for its beautiful beaches, San Diego offers vibrant waterfront attractions, world-class zoos, and stunning sunsets.

2. **Denver, Colorado**
   - The Mile High City boasts outdoor adventures like hiking in the Rockies, a lively downtown scene, and rich cultural experiences.

3. **Seattle, Washington**
   - Famous for its coffee culture, iconic landmarks like the Space Needle, and beautiful waterfront parks, Seattle is a hub for outdoor enthusiasts.


Creating a Bedrock Prompt

In [7]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            { 
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": { 
                    "chat": {
                        'system': [ 
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [ 
                                {
                                    'text': "Trip request: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables' : [
                            { 'name' : 'event_request'}
                        ]
                    }
                }
        }]
    )
    print("Created!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Created!


'arn:aws:bedrock:ap-south-1:567886497519:prompt/10RSUN4LHH'

In [17]:
import boto3

REGION = "us-east-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

MODEL_ID = "us.amazon.nova-micro-v1:0"

# 1) create prompt
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            {
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": {
                    "chat": {
                        "system": [
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries.
Review the message carefully and answer YES or NO to the following screening questions.
Be strict—if any detail is missing or unclear, answer NO.

A) Is the destination clearly stated?
B) Are the travel dates within a reasonable range (not last-minute or over a year away)?
C) Does the request avoid high-risk or restricted activities (e.g., extreme sports, off-grid travel)?
D) Is there any mention of a valid passport or travel documentation?
E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        "messages": [
                            {
                                "role": "user",
                                "content": [
                                    {"text": "Trip request: {{event_request}}"}
                                ]
                            }
                        ],
                        "inputVariables": [
                            {"name": "event_request"}
                        ]
                    }
                }
            }
        ]
    )
    prompt_arn = response["arn"]
    print("Created:", prompt_arn)

except bedrock_agent.exceptions.ConflictException:
    print("Already exists")
    response = bedrock_agent.list_prompts()
    prompt = next(
        (p for p in response["promptSummaries"] if p["name"] == "Travel-Agent-Prompt"),
        None
    )
    prompt_arn = prompt["arn"]
    print("Found existing prompt:", prompt_arn)

# 2) create prompt version
version_response = bedrock_agent.create_prompt_version(
    promptIdentifier=prompt_arn,
    description="v1"
)

prompt_version_arn = version_response["arn"]
print("Prompt version ARN:", prompt_version_arn)

# 3) invoke the PROMPT VERSION ARN, not prompt_arn
response = bedrock.converse(
    modelId=prompt_version_arn,
    promptVariables={
        "event_request": {
            "text": """Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!"""
        }
    }
)

print(response["output"]["message"]["content"][0]["text"])

Created: arn:aws:bedrock:us-east-1:567886497519:prompt/WBXDJ61DBM
Prompt version ARN: arn:aws:bedrock:us-east-1:567886497519:prompt/WBXDJ61DBM:1
A) Yes, the destination is clearly stated. The request is for a trip to Italy.

B) Yes, the travel dates are within a reasonable range. The proposed dates are September 10–20 this year.

C) Yes, the request avoids high-risk or restricted activities. The travelers are looking for a relaxing and enriching experience without extreme sports or off-grid travel.

D) Yes, there is a mention of valid passports. Both travelers have valid passports.

E) Yes, there is enough information to follow up with a proposed itinerary. The destination, travel dates, specific cities to visit, type of experience desired, and travel documentation are all provided.

Therefore, the answer to all screening questions is YES.


In [19]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Hi there! I'm planning a trip to India with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

A) **Is the destination clearly stated?**
   - Yes, the destination is clearly stated as India.

B) **Are the travel dates within a reasonable range (not last-minute or over a year away)?**
   - Yes, the travel dates are within a reasonable range (September 10–20 this year).

C) **Does the request avoid high-risk or restricted activities (e.g., extreme sports, off-grid travel)?**
   - Yes, the request specifies no interest in risky activities like skydiving or remote hiking.

D) **Is there any mention of a valid passport or travel documentation?**
   - Yes, both individuals have valid passports.

E) **Is there enough information to follow up with a proposed itinerary?**
   - Yes, there is enough information provided to start drafting an itinerary, including the destination, travel dates, preferred activities (cultural sites, tours, local restaurants), and avoidance of high-risk activities.

**Final Answer: YES**
